In [ ]:
#Setup Environment and Codebase

import os
import sys

# 2. Define your GitHub Repository
# IMPORTANT: Replace with your actual GitHub repository URL
REPO_URL = "https://github.com/phos-x/retail-model-drift.git"
REPO_NAME = REPO_URL.split("/")[-1].replace(".git", "")

if not os.path.exists(REPO_NAME):
    !git clone {REPO_URL}
    print(f"Successfully cloned {REPO_NAME}")
else:
    print(f"Repository {REPO_NAME} already exists. Pulling latest changes...")
    !cd {REPO_NAME} && git checkout origin dev && git pull

os.chdir(REPO_NAME)

if os.getcwd() not in sys.path:
    sys.path.append(os.getcwd())

print(f"Current Working Directory: {os.getcwd()}")

Current Working Directory: /workspaces/retail-model-drift


In [ ]:
# Download and Format Dataset
import urllib.request
import pandas as pd
from pathlib import Path

Path("data/raw").mkdir(parents=True, exist_ok=True)
Path("data/processed").mkdir(parents=True, exist_ok=True)

csv_path = Path("data/raw/online_retail_II.csv")
excel_url = "https://archive.ics.uci.edu/ml/machine-learning-databases/00502/online_retail_II.xlsx"
excel_path = Path("data/raw/online_retail_II.xlsx")

if not csv_path.exists():
    print("Downloading dataset from UCI Machine Learning Repository...")
    urllib.request.urlretrieve(excel_url, excel_path)
    
    print("Converting Excel to CSV (this may take a minute)...")
    # Read the first sheet (or both and concat, depending on your scope)
    df_excel = pd.read_excel(excel_path, sheet_name="Year 2010-2011") 
    df_excel.to_csv(csv_path, index=False)
    
    # Clean up Excel file to save disk space
    excel_path.unlink()
    print("Dataset ready at:", csv_path)
else:
    print("Dataset already exists at:", csv_path)

In [ ]:

import mlflow
import pandas as pd

from src.data_ingest import load_data
from src.preprocess import clean_and_slice_data, inject_drift
from src.features import compute_rfm_features, assign_targets
from src.train import train_challenger, train_baseline_rf, save_feature_importance_artifact
from src.drift import extract_quantile_bins, compute_psi_report, check_drift_trigger, compute_wasserstein_distance
from src.retrain import assemble_training_window, execute_challenger_retraining
from src.evaluate import compute_residual_stability, evaluate_model_bootstrap, compare_models
from src.mlflow_utils import log_run_to_mlflow

mlflow.set_tracking_uri("sqlite:///mlflow.db") # Local tracking server
mlflow.set_experiment("retail_mlops_champion_challenger")

CONFIG = {
    "random_seed": 42,
    "target_percentile": 75,
    "psi_bins": 10,
    "psi_min_customers": 50,
    "psi_min_bin_count": 5,
    "psi_trigger_threshold": 0.25,
    "auc_drop_tolerance": 0.03,
    "promotion_auc_improvement": 0.02
}

INFO:matplotlib.font_manager:generated new fontManager


In [5]:
# Execute Data Pipeline

raw_df = load_data("data/raw/online_retail_II.csv")

slices_dict = clean_and_slice_data(raw_df, num_slices=6)

rfm_slices = {}
for slice_num, slice_df in slices_dict.items():
    rfm_slices[slice_num] = compute_rfm_features(slice_df)

for i in range(1, 6): 
    rfm_slices[i] = assign_targets(
        current_rfm=rfm_slices[i], 
        next_rfm=rfm_slices[i+1], 
        percentile=CONFIG["target_percentile"]
    )
    print(f"Targets assigned for Slice {i}")

INFO:src.data_ingest:Loading data from data/raw/online_retail_II.csv...
/workspaces/retail-model-drift/retail/lib/python3.12/site-packages/numpy/_core/fromnumeric.py:54: FutureWarning: 'DataFrame.swapaxes' is deprecated and will be removed in a future version. Please use 'DataFrame.transpose' instead.
  return bound(*args, **kwds)
INFO:src.preprocess:Slice 1 created: 67799 records.
INFO:src.preprocess:Slice 2 created: 67799 records.
INFO:src.preprocess:Slice 3 created: 67798 records.
INFO:src.preprocess:Slice 4 created: 67798 records.
INFO:src.preprocess:Slice 5 created: 67798 records.
INFO:src.preprocess:Slice 6 created: 67798 records.


Targets assigned for Slice 1
Targets assigned for Slice 2
Targets assigned for Slice 3
Targets assigned for Slice 4
Targets assigned for Slice 5


In [14]:
# Train initial Champion Model (Slice 1)

baseline_data = rfm_slices[1]

champion_model, baseline_metrics = train_baseline_rf(
    baseline_data, 
    random_seed=CONFIG["random_seed"]
)

baseline_bins = extract_quantile_bins(
    baseline_data, 
    feature_cols=['recency', 'frequency', 'monetary'], 
    num_bins=CONFIG["psi_bins"]
)

# Log baseline
champion_run_id = log_run_to_mlflow(
    run_role="baseline_champion",
    slice_number=1,
    model_pipeline=champion_model,
    metrics=baseline_metrics,
    params=CONFIG
)

print(f"Baseline Champion trained. Validation AUC: {baseline_metrics['val_auc']:.4f}")

INFO:src.train:Fitting Random Forest pipeline...
INFO:src.mlflow_utils:Logging baseline_champion for Slice 1 to MLflow...
2026/04/22 22:37:32 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
INFO:src.mlflow_utils:Successfully logged run. ID: 7a4ffd70c5c74bb38e1cbb3c2b410a20


Baseline Champion trained. Validation AUC: 0.7878


In [16]:
#MLOps Production Simulation (Slices 2 to 5)
# Note: We stop at 5 because Slice 6 lacks a target (Slice 7 doesn't exist)

historical_champion_auc = baseline_metrics['val_auc']

for slice_num in range(2, 6):
    print(f"\n--- Evaluating Slice {slice_num} ---")
    current_data = rfm_slices[slice_num]
    
    # 1. Distributional Monitoring
    psi_report = compute_psi_report(
        baseline_bins, 
        current_data, 
        min_customers=CONFIG["psi_min_customers"],
        min_bin_count=CONFIG["psi_min_bin_count"]
    )
    print(f"Max PSI: {max(psi_report['features'].values()):.4f}")
    
    # 2. Evaluate current Champion
    champion_eval_metrics = evaluate_model_bootstrap(champion_model, current_data)
    current_auc = champion_eval_metrics['auc']
    auc_drop = historical_champion_auc - current_auc
    
    print(f"Champion AUC: {current_auc:.4f} (Drop vs Historical: {auc_drop:.4f})")
    
    # 3. Check Corroborated Trigger
    is_triggered = check_drift_trigger(
        psi_report, 
        auc_drop, 
        psi_threshold=CONFIG["psi_trigger_threshold"],
        auc_tolerance=CONFIG["auc_drop_tolerance"]
    )
    
    if is_triggered:
        print(">> Corroborated Drift Detected! Training Challenger...")
        
        # Train Challenger cumulatively
        training_data = pd.concat([rfm_slices[i] for i in range(1, slice_num)])
        challenger_model, chal_train_metrics = train_challenger(
            training_data, 
            random_seed=CONFIG["random_seed"]
        )
        
        challenger_eval_metrics = evaluate_model_bootstrap(challenger_model, current_data)
        
        # Compare models
        promote, comp_report = compare_models(
            champion_eval_metrics, 
            challenger_eval_metrics,
            min_improvement=CONFIG["promotion_auc_improvement"]
        )
        
        if promote:
            print(f">> Challenger Promoted! New AUC: {challenger_eval_metrics['auc']:.4f}")
            champion_model = challenger_model
            historical_champion_auc = challenger_eval_metrics['auc']
            run_role = "promoted_champion"
            metrics_to_log = challenger_eval_metrics
        else:
            print(">> Challenger rejected (insufficient improvement).")
            run_role = "rejected_challenger"
            metrics_to_log = challenger_eval_metrics
            
        log_run_to_mlflow(
            run_role=run_role, slice_number=slice_num, model=challenger_model,
            metrics=metrics_to_log, psi_report=psi_report, drift_trigger=True, promotion=promote
        )
    else:
        print(">> System stable. No retraining required.")
        log_run_to_mlflow(
            run_role="champion_evaluation", slice_number=slice_num, model_pipeline=champion_model,
            metrics=champion_eval_metrics, psi_report=psi_report, drift_trigger=False, promotion=False
        )

INFO:src.evaluate:Running 1000 bootstrap resamples for AUC confidence intervals...



--- Evaluating Slice 2 ---
Max PSI: 1.3402


INFO:src.drift:Drift trigger suppressed: PSI report flagged as statistically unreliable.
INFO:src.mlflow_utils:Logging champion_evaluation for Slice 2 to MLflow...
2026/04/22 22:39:09 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


Champion AUC: 0.8292 (Drop vs Historical: -0.0415)
>> System stable. No retraining required.


INFO:src.mlflow_utils:Successfully logged run. ID: cc5c1a5b2406488d9d45570fb9321b79
INFO:src.evaluate:Running 1000 bootstrap resamples for AUC confidence intervals...


...

INFO:src.mlflow_utils:Successfully logged run. ID: ddbffc46d0aa4d45b2c5705b2b02ae52
INFO:src.evaluate:Running 1000 bootstrap resamples for AUC confidence intervals...



--- Evaluating Slice 5 ---
Max PSI: 3.8106


INFO:src.drift:Drift trigger suppressed: PSI report flagged as statistically unreliable.
INFO:src.mlflow_utils:Logging champion_evaluation for Slice 5 to MLflow...
2026/04/22 22:39:32 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


Champion AUC: 0.7776 (Drop vs Historical: 0.0102)
>> System stable. No retraining required.


INFO:src.mlflow_utils:Successfully logged run. ID: cc19f798a9cb409091b1092a5c24c1c4


In [ ]:
# Backup MLflow Tracking Data
import shutil
from google.colab import files

print("Zipping MLflow database and artifacts...")
shutil.make_archive("mlops_artifacts", "zip", "mlruns")

# Download the SQLite DB and the zipped artifacts folder
files.download("mlflow.db")
files.download("mlops_artifacts.zip")
print("Downloads initiated. Please save these for your project submission appendix.")

In [ ]:
# Generate Chapter 6 Markdown Reports
import mlflow
import pandas as pd
from IPython.display import display, Markdown

def generate_slice_report(slice_num: int):
    """Queries MLflow and formats the results into the Chapter 6.5 specification."""
    
    exp = mlflow.get_experiment_by_name("retail_mlops_champion_challenger")
    runs_df = mlflow.search_runs(experiment_ids=[exp.experiment_id])
    
    if runs_df.empty:
        return "No runs found. Did the pipeline execute?"

    slice_runs = runs_df[runs_df["params.slice_number"] == str(slice_num)]
    
    if slice_runs.empty:
        return f"No data for Slice {slice_num}."

    baseline_eval = slice_runs[slice_runs["tags.run_role"] == "champion_evaluation"]
    
    promoted_chal = slice_runs[slice_runs["tags.run_role"] == "promoted_champion"]
    
    # 3. Build the Markdown Table Header
    md_table = f"### Slice {slice_num} Performance Report\n\n"
    md_table += "| **Model** | **AUC** | **Run ID (Audit)** | **Promotion** |\n"
    md_table += "|---|---|---|---|\n"
    
    def get_metric(df, metric_name):
        return f"{df.iloc[0][f'metrics.{metric_name}']:.4f}" if not df.empty and f'metrics.{metric_name}' in df.columns else "N/A"
    
    def get_run_id(df):
        return df.iloc[0]['run_id'][:8] if not df.empty else "N/A"

    if not baseline_eval.empty:
        auc = get_metric(baseline_eval, 'auc')
        run_id = get_run_id(baseline_eval)
        md_table += f"| Baseline / Incumbent | {auc} | `{run_id}` | — |\n"
        
    if not promoted_chal.empty:
        auc = get_metric(promoted_chal, 'auc')
        run_id = get_run_id(promoted_chal)
        md_table += f"| Challenger (Adaptive) | **{auc}** | `{run_id}` | **Yes** |\n"
    elif slice_runs[slice_runs["tags.run_role"] == "rejected_challenger"].empty == False:
        rej_chal = slice_runs[slice_runs["tags.run_role"] == "rejected_challenger"]
        auc = get_metric(rej_chal, 'auc')
        run_id = get_run_id(rej_chal)
        md_table += f"| Challenger (Rejected) | {auc} | `{run_id}` | No |\n"
        
    return md_table

print("Generating Audit Reports for Dissertation...\n")
for i in range(2, 6):
    report_md = generate_slice_report(i)
    display(Markdown(report_md))
